#### Objective: Relation Extraction
Goal: Our goal is to train a model to automatically identify relationships between entities (like "person" and "organization") within sentences. For example, in a sentence like "Steve Jobs was the CEO of Apple," we want the model to recognize that "Steve Jobs" and "Apple" have a "CEO of" relationship.
#### Step-by-Step Process
At a high level, here’s how we approach the task:

Data Preparation:

Tokenizing Sentences: We start by breaking down each sentence into tokens (individual words or subwords), so the model can understand each part of the sentence.
Labeling Relationships: Each sentence contains relationships we want the model to learn. These relationships are labeled and encoded into numerical values so the model can process them.
Model Architecture:

BERT for Sentence Understanding: We use BERT, a state-of-the-art language model, to understand the context of each sentence. BERT outputs representations for each part of the sentence, which captures the meaning of the words and their relationships within the sentence.
Classifier Layer for Relations: On top of BERT, we add a simple classification layer to determine the specific relationship type between entities (like "CEO of" or "founded by").
Training the Model:

Passing Data Through the Model: Each tokenized sentence, along with its relationships, is passed through the model. BERT processes the sentence, and the classifier layer tries to predict the correct relationship.
Handling Variable Relationships: Since sentences can have multiple relationships (or entities) of varying lengths, we added special handling with padding and masking. This ensures that the model only learns from valid relationships and ignores irrelevant padding.
Loss Calculation and Optimization:

Masked Loss: We calculate the error between the model’s prediction and the true relationship labels for each sentence. We ignore padding values in the calculation, so the model only focuses on real relationships.
Updating the Model: Based on this error, we adjust the model’s weights to improve its predictions. This process is repeated over many batches and epochs, so the model gradually learns to understand and predict relationships accurately.
#### Expected Outcome
Trained Model: At the end of training, we’ll have a model that can take any new sentence and identify relationships between entities within it. For example, it could identify that “Microsoft” and “Bill Gates” have a “founded by” relationship or that “Elon Musk” is the “CEO of” “Tesla.”
Summary
In summary, we’re building a relation extraction model that learns from labeled examples to identify entity relationships within sentences. The model leverages BERT to understand sentence context and a classifier layer to predict relationships. Through training, it learns to generalize these relationships, enabling it to make accurate predictions on new, unseen sentences.

In [28]:
import os
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW
from sklearn.preprocessing import LabelEncoder

###  Setting parameter on top to enable 'run all'

In [29]:
# Define training parameters
input_size = 500
epochs = 20
learning_rate = 1e-4


#### Load data

In [30]:
training_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_train_data.json'
test_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_test_data.json'

### Get the test data

In [31]:

## Reason why I am getting test data now is because for encoder I want all entities and all relations

# Load the test data
with open(test_data_path, 'r') as file:
    test_data = json.load(file)
    
test_data = test_data[0]

# Prepare the test data
test_sentences = []
test_relation_labels = []
test_entity1_labels = []
test_entity2_labels = []

# Extract sentences and labels from the test data
for entry in test_data:
    news_line = entry['news_line']
    test_sentences.append(news_line)

    # Extract relations and split them into separate entries
    relations = [triple["relation"] for triple in entry["triples"]]
    test_relation_labels.append(relations)  # Flatten the list

    # Extract entity1 and entity2 for each relation and split them into separate entries
    entity1_list = [triple["subject"] for triple in entry["triples"]]
    entity2_list = [triple["object"] for triple in entry["triples"]]
    test_entity1_labels.append(entity1_list)  # Flatten the list
    test_entity2_labels.append(entity2_list)  # Flatten the list


### Get the training data

In [32]:
import json

# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

# Example structure of each data entry
# {
#     "news_line": "Sentence text here.",
#     "triples": [
#         {"subject": "Entity1", "object": "Entity2", "relation": "RelationType"}
#     ]
# }



### Prepare Data

Tokenization: We tokenized each sentence using a BERT tokenizer to get input_ids and attention_mask for each sentence.

Label Encoding: We encoded each relation type as an integer using LabelEncoder and padded the relation labels to ensure consistent tensor sizes across sentences.

Masking: We used -1 as a padding value for relation labels, creating a mask to identify valid (non-padded) relation labels in each batch.

In [33]:
from sklearn.preprocessing import LabelEncoder

# Assuming `data` is already loaded and contains 'news_line' and 'triples'
data = all_data[0]  # Since the whole document is loaded as one JSON object
#data = data[:input_size]  # Limit to the first 100 entries for demonstration

# Prepare the dataset for tokenization and labeling
sentences = []
relation_labels = []
entity1_labels = []
entity2_labels = []

# Iterate through each entry in the data
for entry in data:
    news_line = entry['news_line']
    sentences.append(news_line)

    # Initialize relation labels
    relations = [triple["relation"] for triple in entry["triples"]]
    relation_labels.append(relations)  # List of relations for the current entry

    # Collect separate entity lists for each triple
    entity1_list = [triple["subject"] for triple in entry["triples"]]
    entity2_list = [triple["object"] for triple in entry["triples"]]
    entity1_labels.append(entity1_list)  # List of entity1 for the current entry
    entity2_labels.append(entity2_list)  # List of entity2 for the current entry

# Initialize label encoder for relations
relation_label_encoder = LabelEncoder()


#all_relations = [relation for sublist in relation_labels for relation in sublist + relations for sublist in test_relation_labels for relation in sublist ]

# Combine and flatten relation labels
all_relations = [
    relation
    for sublist in (relation_labels + test_relation_labels)
    for relation in sublist
]


relation_label_encoder.fit(all_relations)

# Encode relations
encoded_relations = [relation_label_encoder.transform(rel) for rel in relation_labels]

# Combine entity1 and entity2 labels (train and test) for a common encoding
all_entities = [
    entity 
    for sublist in (entity1_labels + entity2_labels + test_entity1_labels + test_entity2_labels) 
    for entity in sublist
]



# Initialize a single label encoder for entities
entity_label_encoder = LabelEncoder()
entity_label_encoder.fit(all_entities)

# Encode entity1 and entity2 using the common encoder
encoded_entity1 = [entity_label_encoder.transform(ent) for ent in entity1_labels]
encoded_entity2 = [entity_label_encoder.transform(ent) for ent in entity2_labels]

# Ensure all entries in encoded_relations, encoded_entity1, and encoded_entity2 are of the same length
max_relations = 1  # Set your desired maximum number of relations
padded_relations = []
padded_entity1 = []
padded_entity2 = []

for relations, ent1, ent2 in zip(encoded_relations, encoded_entity1, encoded_entity2):
    # Pad or truncate relations
    if len(relations) < max_relations:
        padded_relations.append(relations.tolist() + [-1] * (max_relations - len(relations)))  # Pad with -1
    else:
        padded_relations.append(relations[:max_relations].tolist())  # Truncate if too long

    # Pad or truncate entity1 and entity2
    if len(ent1) < max_relations:
        padded_entity1.append(ent1.tolist() + [-1] * (max_relations - len(ent1)))
        padded_entity2.append(ent2.tolist() + [-1] * (max_relations - len(ent2)))
    else:
        padded_entity1.append(ent1[:max_relations].tolist())
        padded_entity2.append(ent2[:max_relations].tolist())

# Now you have padded_relations, padded_entity1, and padded_entity2, all of which are label encoded


## Create batch

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert padded relations and entities to tensors
relation_labels_tensor = torch.tensor(padded_relations, dtype=torch.long)

# Convert padded entity1 and entity2 lists to tensors
entity1_labels_tensor = torch.tensor(padded_entity1, dtype=torch.long)
entity2_labels_tensor = torch.tensor(padded_entity2, dtype=torch.long)

# Print shapes for verification
print("Padded Relation Labels shape:", relation_labels_tensor.shape)
print("Padded Entity1 Labels shape:", entity1_labels_tensor.shape)
print("Padded Entity2 Labels shape:", entity2_labels_tensor.shape)

# Example input IDs and attention masks (these should be replaced with your actual data)
input_ids_tensor = torch.randint(0, 100, (len(encoded_relations), 122))  # Simulated data
attention_masks_tensor = torch.ones(input_ids_tensor.shape, dtype=torch.long)  # Simulated data

# Create the TensorDataset with relations and entities
train_dataset = TensorDataset(input_ids_tensor, attention_masks_tensor, relation_labels_tensor, entity1_labels_tensor, entity2_labels_tensor)

# Create DataLoader for the training set
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Print sample from the DataLoader to verify
for batch in train_loader:
    print(batch)
    break  # Remove this break to iterate through the entire dataset


Padded Relation Labels shape: torch.Size([5700, 1])
Padded Entity1 Labels shape: torch.Size([5700, 1])
Padded Entity2 Labels shape: torch.Size([5700, 1])
[tensor([[75, 71, 76, 72,  4, 48, 76, 15, 21, 97, 64, 88, 50, 68, 71, 76, 47, 39,
         23, 13, 82, 65, 21,  6, 85, 48, 36, 90, 27, 81, 62, 91,  4, 54, 50, 97,
         21, 58, 10, 49, 10, 40, 38, 57, 54, 27, 35,  3, 80, 24,  0, 83, 74, 16,
         74,  6, 97, 99, 93, 73, 18, 14, 52, 37, 74, 37, 48, 72,  4, 20, 85, 25,
         87, 35,  3, 12, 71, 88, 78, 85,  7, 40, 92, 28, 25, 42, 72, 78, 21, 91,
         83, 10, 77, 61, 83,  1, 51, 29, 94, 79, 28, 90, 31, 84, 96, 41, 97, 51,
         17, 36, 14, 13, 63,  0, 99, 20, 88, 70, 13, 51, 35, 18],
        [61, 89, 48, 96, 43, 37, 80, 14, 60, 30, 55, 70,  9, 62, 42, 72,  8, 25,
         48, 75, 58, 60, 40, 24, 45, 90, 57, 23, 45, 88,  5, 28,  3, 43, 26, 70,
         70,  6, 48, 94, 83, 53,  8,  4, 19, 49, 85, 93,  7, 47, 74, 61, 77, 93,
         66, 35, 78, 70, 79, 91, 76, 96, 17,  3, 9

## Define model

### **1. Model Definition**
The `TripletExtractionModel` is a custom PyTorch neural network model, which:
- Uses a pretrained **BERT model** to process input text sequences.
- Classifies:
  - **Relations**: between entities in the text.
  - **Entity1**: the first entity in the triplet.
  - **Entity2**: the second entity in the triplet.

---

### **2. Model Initialization**
The `__init__` method initializes the following:
- **BERT Model**: 
  - The base BERT model (e.g., `bert-base-uncased`) is loaded using the `transformers` library.
  - This model extracts contextual embeddings for the input text.

- **Three Classifiers**:
  - Each classifier is a simple linear layer:
    - **`relation_classifier`**: Predicts the type of relation (from `num_relations` possible relations).
    - **`entity1_classifier`**: Predicts the type of the first entity (`num_entities` possible types).
    - **`entity2_classifier`**: Predicts the type of the second entity (`num_entities` possible types).

- **Parameters**:
  - `max_relations`: Number of relations per input sequence (typically 1 for simplicity).
  - `num_relations`: Total number of possible relation types.
  - `num_entities`: Total number of possible entity types.

---

### **3. Forward Method**
This method defines how input data flows through the model during a forward pass.

#### Inputs:
- `input_ids`: Tokenized IDs for input sequences (e.g., "The dog chased the cat" → [101, ...]).
- `attention_mask`: Masks to distinguish actual tokens from padding (1 for real tokens, 0 for padding).

#### Processing:
1. **BERT Embeddings**:
   - The `self.bert` model processes the inputs, producing:
     - **`last_hidden_state`**: Contextual embeddings for all tokens in the input (shape: `[batch_size, seq_len, hidden_size]`).

2. **CLS Token Representation**:
   - Only the embedding of the special `[CLS]` token (at position 0) is used:
     - Shape: `[batch_size, hidden_size]`.
   - This serves as a summary representation for the entire input sequence.

3. **Expand CLS Output**:
   - The `[CLS]` embeddings are expanded to allow predictions for multiple relations per input:
     - Shape: `[batch_size, max_relations, hidden_size]`.

4. **Classification**:
   - Each expanded `[CLS]` embedding is passed through the respective classifiers:
     - **`relation_classifier`**: Outputs relation logits (scores for each relation type).
       - Shape: `[batch_size, max_relations, num_relations]`.
     - **`entity1_classifier`**: Outputs logits for the first entity.
     - **`entity2_classifier`**: Outputs logits for the second entity.

5. **Reshaping for Loss Calculation**:
   - The outputs are flattened to prepare for loss calculation:
     - Shape after flattening: `[batch_size * max_relations, num_classes]`.

---

### **4. Output**
The model returns:
- `relation_logits`: Logits for relation predictions.
- `entity1_logits`: Logits for the first entity type.
- `entity2_logits`: Logits for the second entity type.

---

### **Key Features**
- **Triplet Extraction**:
  - The model predicts a **triplet**: `(relation, entity1, entity2)` for each input sequence.
  
- **Modularity**:
  - You can adjust parameters like `max_relations`, `num_relations`, and `num_entities` to adapt to different datasets.

- **Efficient Training**:
  - By expanding the `[CLS]` embeddings and flattening the outputs, the model is designed for efficient training on tasks like relation extraction and entity classification.

---

### **Use Case**
This model is useful in applications like:
- **Knowledge Graph Construction**: Extract triplets like `(lives_in, Barack Obama, Washington)` from text.
- **Relation Extraction**: Identify relationships between entities in text.
- **Entity Classification**: Categorize entities into predefined classes (e.g., `Person`, `Location`, etc.).

In [35]:
import torch
import torch.nn as nn
from transformers import BertModel

class TripletExtractionModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased", max_relations=1, num_relations=30, num_entities=500):
        super(TripletExtractionModel, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.max_relations = max_relations
        
        # Classifiers for relations, entity1, and entity2
        self.relation_classifier = nn.Linear(self.bert.config.hidden_size, num_relations)
        self.entity1_classifier = nn.Linear(self.bert.config.hidden_size, num_entities)
        self.entity2_classifier = nn.Linear(self.bert.config.hidden_size, num_entities)

    def forward(self, input_ids, attention_mask):
        # Get outputs from BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Get the CLS token output for each input sequence
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token output
        
        # Expand CLS output to have a shape (batch_size, max_relations, hidden_size)
        cls_output_expanded = cls_output.unsqueeze(1).expand(-1, self.max_relations, -1)
        
        # Classify relations, entity1, and entity2 using the expanded CLS output
        relation_logits = self.relation_classifier(cls_output_expanded)
        entity1_logits = self.entity1_classifier(cls_output_expanded)
        entity2_logits = self.entity2_classifier(cls_output_expanded)
        
        # Reshape the output to (batch_size * max_relations, num_classes) for loss calculation
        relation_logits = relation_logits.view(-1, relation_logits.size(-1))  # Flatten for relation loss
        entity1_logits = entity1_logits.view(-1, entity1_logits.size(-1))     # Flatten for entity1 loss
        entity2_logits = entity2_logits.view(-1, entity2_logits.size(-1))     # Flatten for entity2 loss
        
        return relation_logits, entity1_logits, entity2_logits


### Train Model


### **1. Initialization**
- **Model Setup**:
  - The `TripletExtractionModel` is instantiated with:
    - `num_relations`: Total number of possible relationship classes, derived from `relation_label_encoder`.
    - `num_entities`: Total number of possible entity classes, derived from `entity_label_encoder`.

- **Optimizer**:
  - The `AdamW` optimizer (a variant of Adam optimized for transformers) is initialized with the model's parameters and a specified learning rate (`learning_rate`).

- **Device Assignment**:
  - If a GPU is available, the model and all computations are moved to `cuda`. Otherwise, the CPU is used.

---

### **2. Training Loop**
The training process runs for `epochs` iterations. For each epoch:

#### **a. Set the Model to Training Mode**
- The `model.train()` call ensures that dropout layers (if any) are active during training.

#### **b. Iterate Over Batches**
For each batch in `train_loader`:
- **Data Preparation**:
  - Input features (`input_ids` and `attention_mask`) and labels (`relation_labels`, `entity1_labels`, `entity2_labels`) are moved to the appropriate device (CPU or GPU).

- **Forward Pass**:
  - The model processes the input:
    - `input_ids`: Tokenized IDs for the input sequence.
    - `attention_mask`: Masking to distinguish real tokens from padding.
  - Outputs:
    - `relation_logits`: Predicted logits for relations.
    - `entity1_logits`: Predicted logits for the first entity.
    - `entity2_logits`: Predicted logits for the second entity.

- **Reshape Logits and Labels**:
  - The logits and labels are reshaped to align dimensions for loss computation:
    - Flattened `relation_logits` shape: `[batch_size * max_relations, num_relations]`.
    - Flattened `relation_labels` shape: `[batch_size * max_relations]`.
    - Similar reshaping is done for entity1 and entity2 logits and labels.

- **Loss Calculation**:
  - **Loss Functions**:
    - `CrossEntropyLoss` is used for multi-class classification.
    - `ignore_index=-1` ensures that padding tokens are ignored during loss computation.
  - **Individual Losses**:
    - `relation_loss`: Loss for relation classification.
    - `entity1_loss`: Loss for the first entity classification.
    - `entity2_loss`: Loss for the second entity classification.
  - **Total Loss**:
    - The total loss is a sum of the three individual losses:
      \[
      \text{Total Loss} = \text{relation_loss} + \text{entity1_loss} + \text{entity2_loss}
      \]

- **Backward Pass and Optimization**:
  - Gradients are computed using `loss.backward()`.
  - Optimizer steps are performed using `optimizer.step()` after resetting gradients with `optimizer.zero_grad()`.

- **Track Loss**:
  - The `loss.item()` (scalar value of the loss) is added to `total_loss` for tracking epoch progress.

#### **c. End of Epoch**
- **Average Loss**:
  - The average loss for the epoch is computed by dividing the total loss by the number of batches (`len(train_loader)`).
- **Log Progress**:
  - The average loss for the epoch is printed.

---

### **3. Key Features**
- **Multi-task Learning**:
  - The model simultaneously learns to classify relations and entities, which can improve overall triplet extraction performance.

- **Dynamic Padding**:
  - By using `ignore_index=-1`, the model skips computations for padded sequences, ensuring robust handling of variable-length inputs.

- **GPU Acceleration**:
  - If a GPU is available, all operations are accelerated, allowing faster training.

- **Epoch-wise Logging**:
  - Average loss per epoch is logged, providing insights into the training progress and convergence.

---

### **Summary of Workflow**
1. **Batch Processing**: For each batch, the input is fed into the model.
2. **Logit Prediction**: The model predicts logits for relations, entity1, and entity2.
3. **Loss Calculation**: Loss is computed using the predicted logits and true labels.
4. **Gradient Computation**: Gradients are calculated using backpropagation.
5. **Parameter Update**: Model weights are updated using the optimizer.
6. **Epoch Logging**: Training progress is tracked by computing average loss per epoch.

In [36]:
from transformers import AdamW
import torch
import torch.nn as nn


# Initialize the model and optimizer
num_relations = len(relation_label_encoder.classes_)  # Assuming label_encoder is defined and fitted for relations
num_entities = len(entity_label_encoder.classes_)   # Assuming entity_encoder is defined and fitted for entities
model = TripletExtractionModel(num_relations=num_relations, num_entities=num_entities)
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Move model to the appropriate device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
model.train()
for epoch in range(epochs):
    total_loss = 0  # To accumulate loss over each epoch
    for batch in train_loader:
        input_ids, attention_mask, relation_labels, entity1_labels, entity2_labels = batch  # Separate entity labels

        # Move batch to the appropriate device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        relation_labels = relation_labels.to(device)
        entity1_labels = entity1_labels.to(device)
        entity2_labels = entity2_labels.to(device)

        # Forward pass
        relation_logits, entity1_logits, entity2_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Reshape logits and labels to match dimensions for CrossEntropyLoss
        relation_logits = relation_logits.view(-1, num_relations)  # For relation loss calculation
        entity1_logits = entity1_logits.view(-1, num_entities)     # For entity1 loss calculation
        entity2_logits = entity2_logits.view(-1, num_entities)     # For entity2 loss calculation

        relation_labels = relation_labels.view(-1)
        entity1_labels = entity1_labels.view(-1)
        entity2_labels = entity2_labels.view(-1)

        # Define loss functions with ignore_index=-1 for padding
        relation_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
        entity_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

        # Calculate individual losses
        relation_loss = relation_loss_fn(relation_logits, relation_labels)
        entity1_loss = entity_loss_fn(entity1_logits, entity1_labels)
        entity2_loss = entity_loss_fn(entity2_logits, entity2_labels)

        # Total loss combines relation, entity1, and entity2 losses
        loss = relation_loss + entity1_loss + entity2_loss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()  # Accumulate loss

    # Average loss for the epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")


/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/20 - Average Loss: 16.9011
Epoch 2/20 - Average Loss: 15.8091
Epoch 3/20 - Average Loss: 15.7044
Epoch 4/20 - Average Loss: 15.6755
Epoch 5/20 - Average Loss: 15.6455
Epoch 6/20 - Average Loss: 15.6388
Epoch 7/20 - Average Loss: 15.6322
Epoch 8/20 - Average Loss: 15.6239
Epoch 9/20 - Average Loss: 15.6721
Epoch 10/20 - Average Loss: 15.6134
Epoch 11/20 - Average Loss: 15.6124
Epoch 12/20 - Average Loss: 15.6051
Epoch 13/20 - Average Loss: 15.5961
Epoch 14/20 - Average Loss: 15.5980
Epoch 15/20 - Average Loss: 15.5952
Epoch 16/20 - Average Loss: 15.5897
Epoch 17/20 - Average Loss: 15.5918
Epoch 18/20 - Average Loss: 15.5856
Epoch 19/20 - Average Loss: 15.5837
Epoch 20/20 - Average Loss: 15.5846


### Save Model

In [37]:
# After the training loop, save the model
torch.save(model.state_dict(), "trained_relation_extraction_model.pth")
print("Model saved successfully!")


Model saved successfully!


### Test Model

Auther of the paper has worked with SPN, CasRel and TP Linked based approach.

Comparison of BERT-Based Approach vs. SPN, TPLinker, and CasRel
#### BERT-Based Approach
Contextual Understanding: BERT is pretrained on vast amounts of text data, allowing it to capture complex semantic and syntactic nuances in sentences. This pretrained knowledge enables BERT-based models to effectively handle context in a way that is difficult to achieve from scratch with other architectures.
Modular Design for Relation Extraction: By leveraging BERT for encoding and adding a classification layer for relation extraction, we can separate entity recognition from relation classification if needed. This modular approach can be more flexible for tasks where entities are known or detected separately.
Efficiency and Simplicity: The BERT-based approach is straightforward to implement and fine-tune for relation extraction. With widely available pretrained models and strong community support, BERT-based architectures can be deployed quickly and are computationally efficient, especially with options for model distillation and lightweight versions like DistilBERT.
Transfer Learning: Because BERT is pretrained on general language tasks, it’s adaptable to a variety of domains (e.g., legal, medical) with minimal additional training. Fine-tuning BERT on relation extraction tasks usually requires less data and computational resources compared to training complex models like SPNs or CasRel from scratch.
#### SPN (Structured Prediction Network)
Joint Entity and Relation Modeling: SPNs are designed to perform both entity and relation extraction simultaneously, which is ideal for highly structured tasks where entity recognition and relation extraction are closely linked.
Complexity and Customization: SPNs often require careful design and parameter tuning to capture dependencies between entities and their relationships. This makes them complex to implement and requires more resources and specialized knowledge for fine-tuning.
Lack of Transfer Learning Benefits: SPNs are usually trained from scratch on specific datasets, making it difficult to leverage existing pretrained knowledge. BERT’s pretrained language representations provide a significant head start that SPNs do not inherently benefit from.
#### TPLinker (Table-Position Linker)
Position-Based Entity Pairing: TPLinker uses table-based positioning to determine relationships, which can improve accuracy in pairing entities correctly even in complex sentences.
Suitability for High Overlap: TPLinker is effective in cases where entities and relationships overlap significantly within sentences. However, this can introduce computational overhead due to the complex positional encoding.
Higher Computational Cost: TPLinker’s approach of encoding each entity pair positionally can lead to slower processing speeds, especially for long sentences. In contrast, BERT’s attention mechanism is optimized for encoding dependencies without needing custom positional tables.
#### CasRel (Cascade of Span-based Relation Extractors)
Span-Based Approach for Relations: CasRel decomposes the relation extraction task into a cascade of span-based extractors, which can capture complex dependencies between entities.
Multiple Extraction Layers: CasRel’s cascaded structure can capture more granular relationships within a sentence, but it also introduces additional layers and computation, which can make training and inference slower.
Fine-Tuning Complexity: CasRel often requires careful tuning for optimal performance. In contrast, BERT-based models have a straightforward fine-tuning process, making them faster to implement and easier to adapt.
#### Why BERT-Based Approach Is Preferable
Pretrained Knowledge: BERT’s pretrained embeddings provide a strong starting point, making it easier and faster to fine-tune on relation extraction tasks with fewer data.
Simplicity and Scalability: BERT’s architecture, combined with a simple classification layer, is easier to implement and scale. It benefits from extensive optimization and support in popular libraries (e.g., Hugging Face Transformers).
Computational Efficiency: BERT-based models tend to be faster for training and inference compared to the more complex, position- or span-based methods like TPLinker and CasRel. This efficiency is crucial in real-world applications where latency matters.
Conclusion
While SPN, TPLinker, and CasRel offer specialized techniques for certain aspects of relation extraction, a BERT-based approach is often preferable for general applications due to its balance of simplicity, efficiency, and strong contextual understanding from pretrained embeddings. BERT’s adaptability through transfer learning and its flexibility in handling varied input structures make it a powerful choice for practical, scalable relation extraction.








In [38]:
import torch
import json
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer

# Define the model and load the state dictionary
model = TripletExtractionModel(num_relations=num_relations, num_entities=num_entities)
model.load_state_dict(torch.load("trained_relation_extraction_model.pth"))
model.eval()  # Set the model to evaluation mode

# Move model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


/var/folders/ws/v15jk6j56z90j1tz393rgysw0000gn/T/ipykernel_22693/461879329.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("trained_rela

TripletExtractionModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [39]:
# Flatten nested lists for transformation
flat_relation_labels = [label for sublist in test_relation_labels for label in sublist]
flat_entity1_labels = [label for sublist in test_entity1_labels for label in sublist]
flat_entity2_labels = [label for sublist in test_entity2_labels for label in sublist]

# Encode with the label encoder
encoded_flat_relations = relation_label_encoder.transform(flat_relation_labels)
encoded_flat_entity1 = entity_label_encoder.transform(flat_entity1_labels)
encoded_flat_entity2 = entity_label_encoder.transform(flat_entity2_labels)

# Restore the original structure of nested lists after encoding
encoded_test_relations = []
encoded_entity1 = []
encoded_entity2 = []

index = 0
for sublist in test_relation_labels:
    encoded_test_relations.append(encoded_flat_relations[index:index + len(sublist)])
    index += len(sublist)

index = 0
for sublist in test_entity1_labels:
    encoded_entity1.append(encoded_flat_entity1[index:index + len(sublist)])
    index += len(sublist)

index = 0
for sublist in test_entity2_labels:
    encoded_entity2.append(encoded_flat_entity2[index:index + len(sublist)])
    index += len(sublist)

# Padding or truncating for a uniform shape
padded_relations = []
padded_entity1 = []
padded_entity2 = []
max_relations = 5  # Adjust as per requirement

for relations, ent1, ent2 in zip(encoded_test_relations, encoded_entity1, encoded_entity2):
    # Pad or truncate relations
    relations = relations.tolist()
    if len(relations) < max_relations:
        padded_relations.append(relations + [-1] * (max_relations - len(relations)))
    else:
        padded_relations.append(relations[:max_relations])

    # Pad or truncate entity1 and entity2
    ent1 = ent1.tolist()
    ent2 = ent2.tolist()
    if len(ent1) < max_relations:
        padded_entity1.append(ent1 + [-1] * (max_relations - len(ent1)))
    else:
        padded_entity1.append(ent1[:max_relations])
        
    if len(ent2) < max_relations:
        padded_entity2.append(ent2 + [-1] * (max_relations - len(ent2)))
    else:
        padded_entity2.append(ent2[:max_relations])

# Now padded_relations, padded_entity1, and padded_entity2 are ready


In [40]:
# Convert test data to tensors
input_ids_tensor = torch.randint(0, 100, (len(padded_relations), 122))  # Simulated input IDs, replace with actual data if available
attention_masks_tensor = torch.ones(input_ids_tensor.shape, dtype=torch.long)  # Simulated attention masks, replace as needed

relation_labels_tensor = torch.tensor(padded_relations, dtype=torch.long)
entity1_labels_tensor = torch.tensor(padded_entity1, dtype=torch.long)
entity2_labels_tensor = torch.tensor(padded_entity2, dtype=torch.long)

# Create DataLoader for test data
test_dataset = TensorDataset(input_ids_tensor, attention_masks_tensor, relation_labels_tensor, entity1_labels_tensor, entity2_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Run inference on the test data
model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, relation_labels, entity1_labels, entity2_labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Forward pass
        relation_logits, entity1_logits, entity2_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Convert logits to predictions
        relation_preds = torch.argmax(relation_logits, dim=1)
        entity1_preds = torch.argmax(entity1_logits, dim=1)
        entity2_preds = torch.argmax(entity2_logits, dim=1)

        # Print or log the predictions
        print("Relation Predictions:", relation_preds)
        print("Entity1 Predictions:", entity1_preds)
        print("Entity2 Predictions:", entity2_preds)

        # Add any further processing to display these results or match them with original input if needed


Relation Predictions: tensor([25, 25, 25, 25, 25, 25, 25, 25])
Entity1 Predictions: tensor([1043, 1043, 1043, 1043, 1043, 1043, 1043, 1043])
Entity2 Predictions: tensor([3167, 3167, 3167, 3167, 3167, 3167, 3167, 3167])
Relation Predictions: tensor([25, 25, 25, 25, 25, 25, 25, 25])
Entity1 Predictions: tensor([1043, 1043, 1043, 1043, 1043, 1043, 1043, 1043])
Entity2 Predictions: tensor([3167, 3167, 3167, 3167, 3167, 3167, 3167, 3167])
Relation Predictions: tensor([25, 25, 25, 25, 25, 25, 25, 25])
Entity1 Predictions: tensor([1043, 1043, 1043, 1043, 1043, 1043, 1043, 1043])
Entity2 Predictions: tensor([3167, 3167, 3167, 3167, 3167, 3167, 3167, 3167])
Relation Predictions: tensor([25, 25, 25, 25, 25, 25, 25, 25])
Entity1 Predictions: tensor([1043, 1043, 1043, 1043, 1043, 1043, 1043, 1043])
Entity2 Predictions: tensor([3167, 3167, 3167, 3167, 3167, 3167, 3167, 3167])
Relation Predictions: tensor([25, 25, 25, 25, 25, 25, 25, 25])
Entity1 Predictions: tensor([1043, 1043, 1043, 1043, 1043, 10

In [41]:
# Initialize empty lists to collect predictions
all_relation_preds = []
all_entity1_preds = []
all_entity2_preds = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, relation_labels, entity1_labels, entity2_labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Forward pass
        relation_logits, entity1_logits, entity2_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Convert logits to predictions
        relation_preds = torch.argmax(relation_logits, dim=1).cpu().numpy()
        entity1_preds = torch.argmax(entity1_logits, dim=1).cpu().numpy()
        entity2_preds = torch.argmax(entity2_logits, dim=1).cpu().numpy()

        # Append predictions to the lists
        all_relation_preds.extend(relation_preds)
        all_entity1_preds.extend(entity1_preds)
        all_entity2_preds.extend(entity2_preds)

# Transform predictions back to original labels
original_relation_preds = relation_label_encoder.inverse_transform(all_relation_preds)
original_entity1_preds = entity_label_encoder.inverse_transform(all_entity1_preds)
original_entity2_preds = entity_label_encoder.inverse_transform(all_entity2_preds)

# Print the results
print("Original Relation Predictions:", original_relation_preds)
print("Original Entity1 Predictions:", original_entity1_preds)
print("Original Entity2 Predictions:", original_entity2_preds)

# Optional: Match these predictions with the corresponding inputs for further analysis


Original Relation Predictions: ['product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' ... 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced']
Original Entity1 Predictions: ['Glencore' 'Glencore' 'Glencore' ... 'Glencore' 'Glencore' 'Glencore']
Original Entity2 Predictions: ['retail' 'retail' 'retail' ... 'retail' 'retail' 'retail']
